# Session 9 lab: The chain rule and the Jacobian

**Mathematics for AI and Machine Learning, KLAi Professional Learning**

Today you compute the slope of a function of a function with the chain rule, run a tiny computation graph forwards and backwards, compute the Jacobian of a function with two inputs and two outputs, and check every answer with finite differences (nudges). The delivery data are the same four illustrative deliveries as Sessions 7 and 8 (made-up numbers): distance x in km, fee y in thousand UGX. The spreadsheet lab uses the same numbers.

Rhythm of every step: **PREDICT**, run, **READ**, **CHANGE one thing**. Run the cells in order from the top. A cell that says `None  # your code here` is yours to fill. Lines that start with `check(` tell you whether your answer is right.

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 0: Set up and write the checking tool

We use numpy for numbers. The function slope() estimates a slope by nudging the input up and down (the central difference from Session 7). We use it to check every answer today.

In [ ]:
import numpy as np

x = np.array([1.0, 2.0, 3.0, 4.0])   # distance in km
y = np.array([3.0, 5.0, 6.0, 9.0])   # fee in thousand UGX

def slope(f, at, h=1e-5):
    # central difference: change in f over a tiny step on both sides, divided by 2h
    return (f(at + h) - f(at - h)) / (2 * h)

print("slope of x squared at 3:", round(slope(lambda t: t ** 2, 3.0), 4))

---
## Step 1: The chain rule on y = (3x + 1) squared

The inner function is u = 3x + 1 and the outer function is y = u squared. The slope of y with respect to x is the outer slope times the inner slope.

**PREDICT first.** At x = 2 the inner value is u = 7. What is the outer slope 2u, and what is the product with the inner slope 3? Write the number down before you run.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def inner(x):
    return 3 * x + 1

def outer(u):
    return u ** 2

x0 = 2.0
u0 = inner(x0)
# Hint: dy_du is the slope of the outer function u squared at u0; du_dx is the slope of the inner function 3x + 1; chain combines the two
dy_du = None  # your code here
du_dx = None  # your code here
chain = None  # your code here
print('u =', u0, ' dy/du =', dy_du, ' du/dx =', du_dx, ' chain rule slope =', chain)
nudged = slope(lambda t: outer(inner(t)), x0)
print('nudged slope:', round(nudged, 4))
check('chain rule slope at x = 2', chain, 42)
check('chain rule slope agrees with the nudged slope', chain, nudged, tol=0.001)

**READ the output.** Which of the two numbers did you forget if you got 14 instead of 42?

> My answer: 

---
## Step 2: A second chain: y = e to the 2x

The inner function is u = 2x with slope 2. The outer function is e to the u, whose slope is e to the u. Evaluate the outer slope at u = 2x.

**PREDICT first.** At x = 1 the inner value is u = 2. Using the table of examples on the slides, what do you expect the chain rule slope to be?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
x1 = 1.0
# Hint: the slope of the inner function 2x, times the slope of the outer function e to the u at u = 2 * x1
chain_exp = None  # your code here
print('chain rule slope:', round(chain_exp, 3))
print('nudged slope    :', round(slope(lambda t: np.exp(2 * t), x1), 3))
check('slope of e to the 2x at x = 1', chain_exp, 14.778, tol=0.001)
check('slope of e to the 2x agrees with the nudged slope', chain_exp, slope(lambda t: np.exp(2 * t), x1), tol=0.001)

**CHANGE one thing.** In a new cell, change the inner function to 5x and recompute at x = 1. Predict the answer first: it should be 5 times e to the 5.

> What I saw: 

---
## Step 3: The tiny graph: forward pass, then backward pass

One delivery: x = 3 km, fee y = 6. The model is p = w x and the loss is L = (p - y) squared, at w = 1. Forward gives values; backward gives slopes, each one the previous slope times a local slope.

**PREDICT first.** Forward: what are p and L? Backward: what is dL/dp = 2 (p - y), and then dL/dw = dL/dp times x?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
xd, yd, w = 3.0, 6.0, 1.0
# Hint: p is the model prediction and L is the squared error of that one delivery
p = None  # your code here
L = None  # your code here
# Hint: dL_dp is the slope of L with respect to p; dL_dw multiplies it by the local slope of p with respect to w
dL_dp = None  # your code here
dL_dw = None  # your code here
print('forward :', p, L)
print('backward:', dL_dp, dL_dw)
nudged_w = slope(lambda t: (t * xd - yd) ** 2, w)
print('nudged dL/dw:', round(nudged_w, 3))
check('forward p', p, 3)
check('forward L', L, 9)
check('backward dL/dp', dL_dp, -6)
check('backward dL/dw', dL_dw, -18)
check('backward dL/dw agrees with the nudged slope', dL_dw, nudged_w, tol=0.001)

**READ the output.** The value dL_dw is the slope of the loss with respect to w. Which way should w move, up or down?

> My answer: 

---
## Step 4: The same graph for all four deliveries

Run the backward pass for each row. The loss of the whole dataset is the mean of the four row losses, so its slope is the mean of the four backward values.

**PREDICT first.** The four errors p minus y at w = 1 are -2, -3, -3 and -5. Guess the mean of the four dL/dw values. Look back at Session 8: what was dL/dw at (1, 0)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
errors = w * x - y
# Hint: row_dp is the slope of each row loss with respect to its prediction; row_dw multiplies it by the local slope of the prediction with respect to w; mean_dw averages the four rows
row_dp = None  # your code here
row_dw = None  # your code here
mean_dw = None  # your code here
print('row dL/dw values:', row_dw)
print('mean            :', mean_dw)
check('row dL/dw values', row_dw, [-4, -12, -18, -40])
check('mean dL/dw equals the Session 8 slope', mean_dw, -18.5)

**CHANGE one thing.** In a new cell set w = 2 and rerun the backward pass for the four rows. Predict first: will the mean be negative, zero or positive? (Answer: -3.5. It is still negative, so w should still go up. The best w is 2.233.)

> What I saw: 

---
## Step 5: The Jacobian of a function with two inputs and two outputs

F(x1, x2) = (u, v) with u = x1 times x2 and v = x1 + x2 squared. The Jacobian has one row for each output and one column for each input.

**PREDICT first.** At the point (2, 3), what are the four entries? Row u: du/dx1 and du/dx2. Row v: dv/dx1 and dv/dx2.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def F(a, b):
    return np.array([a * b, a + b ** 2])

point = (2.0, 3.0)
print("outputs u, v at (2, 3):", F(*point))

# Hint: differentiate u with respect to x1, then x2, for row u; do the same for v. Use point[0] for x1 and point[1] for x2.
J = None  # your code here: a 2 by 2 array
print('Jacobian by formula:')
print(J)
# nudge one input at a time: each nudge gives one whole column
h = 1e-5
col1 = (F(point[0] + h, point[1]) - F(point[0] - h, point[1])) / (2 * h)
col2 = (F(point[0], point[1] + h) - F(point[0], point[1] - h)) / (2 * h)
J_nudged = np.column_stack([col1, col2])
print('Jacobian by nudging:')
print(np.round(J_nudged, 3))
check('Jacobian by formula', J, [[3, 2], [1, 6]])
check('Jacobian by formula agrees with nudging', J, J_nudged, tol=0.001)

**READ the output.** Which entry is the largest, and which input does that output depend on most strongly at this point?

> My answer: 

---
## Step 6: Back to the inputs: J transposed times a gradient

Let s = u times v, a single number computed from the two outputs. Its slopes with respect to u and v are v and u. Multiplying by J transposed brings them back to the inputs.

**PREDICT first.** At the point (2, 3) u = 6 and v = 11, so the slopes of s with respect to u and v are 11 and 6. Guess the sign of the two slopes with respect to x1 and x2.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
u_val, v_val = F(*point)
grad_uv = np.array([v_val, u_val])   # ds/du = v and ds/dv = u
# Hint: the slopes with respect to the inputs are the transposed Jacobian times grad_uv (in numpy, J.T and @)
grad_x = None  # your code here
print('gradient of s in the inputs:', grad_x)
s_of = lambda a, b: F(a, b)[0] * F(a, b)[1]
nudged_s = np.array([slope(lambda t: s_of(t, point[1]), point[0]), slope(lambda t: s_of(point[0], t), point[1])])
print('nudged slopes of s            :', np.round(nudged_s, 3))
check('gradient of s in the inputs', grad_x, [39, 58])
check('gradient of s agrees with the nudged slopes', grad_x, nudged_s, tol=0.001)

**CHANGE one thing.** In a new cell move the point to (1, 2). Predict first with the chain rule, then compute J.T @ grad_uv again. (Answer: the Jacobian is [[2, 1], [1, 4]] and the gradient of s is (12, 13).)

> What I saw: 

---
## Finished

Run the next cell for your automatic check. Then use File, Save. Keep the numbers 42, -18 and the Jacobian [[3, 2], [1, 6]].

In [ ]:
show_checks()